# Milestone 5 — Phase 2C: Hyperparameter Optimization

Tune the three Phase 2A finalists using nested stratified cross-validation. The primary selection metric is ROC-AUC. All tuning and evaluation use only the original training partition.

**Holdout protection:** the historical holdout was already used for final evaluation. This notebook reconstructs the original split only to recover the training rows, immediately discards the holdout, and never passes it to the experiment function.

## 1. Environment and dataset setup

Run in Google Colab. Upload the original IBM Telco Customer Churn CSV if it is not already present.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/adhishmodi/customer-churn-prediction.git"
EXPECTED_CSV = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = Path("/content/customer-churn-prediction")
    if not (PROJECT_ROOT / "src").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_PATH = PROJECT_ROOT / "data" / "raw" / EXPECTED_CSV
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATA_PATH}")


In [ ]:
if not DATA_PATH.exists():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(f"Place the original CSV at {DATA_PATH}") from exc
    print(f"Upload {EXPECTED_CSV}")
    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if EXPECTED_CSV in uploaded:
        selected_name = EXPECTED_CSV
    elif len(csv_names) == 1:
        selected_name = csv_names[0]
    else:
        raise ValueError(f"Upload exactly one CSV. Found: {csv_names}")
    DATA_PATH.write_bytes(uploaded[selected_name])
assert DATA_PATH.exists()


## 2. Reconstruct training data only

Recreate the original stratified 80/20 split with seed 42. The holdout is deleted immediately and is not used for any Phase 2C computation.

In [ ]:
import pandas as pd
from IPython.display import display
from src.preprocess import load_and_clean_data, prepare_features_target, split_data

cleaned_df = load_and_clean_data(DATA_PATH)
X, y = prepare_features_target(cleaned_df)
X_train, discarded_features, y_train, discarded_labels = split_data(X, y)
del discarded_features, discarded_labels, X, y, cleaned_df
print(f"Training rows: {len(X_train):,}")
print(f"Training churn prevalence: {y_train.mean():.2%}")
print("Historical holdout used: False")


## 3. Search protocol

- Candidates: unweighted XGBoost, balanced XGBoost, and balanced Random Forest.
- Outer validation: 5-fold stratified CV.
- Inner search: 3-fold stratified CV.
- Search budget: 12 randomized parameter combinations per candidate per outer fold (540 inner model fits total).
- Primary selection metric: ROC-AUC, fixed before running the experiment.
- Secondary metrics: Average Precision, precision, recall, F1, and accuracy.
- Classification metrics use the estimator's default 0.50 threshold; threshold optimization is outside Phase 2C.

Preprocessing stays inside the pipeline. Class weights are computed within each model fit. The outer validation fold is not involved in parameter search or candidate selection.

In [ ]:
from src.experiments import evaluate_phase_2c_nested_cv

N_OUTER_SPLITS = 5
N_INNER_SPLITS = 3
N_ITER = 12
RANDOM_STATE = 42
print(f"Outer folds: {N_OUTER_SPLITS}")
print(f"Inner folds: {N_INNER_SPLITS}")
print(f"Random trials per candidate per outer fold: {N_ITER}")
print("Expected inner fits: 5 × 3 × 12 × 3 = 540")


## 4. Run nested optimization

This experiment may take several minutes. Do not interrupt the runtime before all five outer folds finish.

In [ ]:
phase_2c_results = evaluate_phase_2c_nested_cv(
    X_train,
    y_train,
    outer_splits=N_OUTER_SPLITS,
    inner_splits=N_INNER_SPLITS,
    n_iter=N_ITER,
    random_state=RANDOM_STATE,
)
assert phase_2c_results["configuration"]["test_set_used"] is False
assert len(phase_2c_results["outer_fold_results"]) == N_OUTER_SPLITS
assert len(phase_2c_results["inner_candidate_results"]) == N_OUTER_SPLITS * 3
print(phase_2c_results["configuration"])
print("Phase 2C nested optimization completed.")


## 5. Review results

The outer-fold summary estimates the complete tuning-and-selection procedure. Do not substitute the best inner-CV score for the outer validation estimate. Fold standard deviation is descriptive variability, not a confidence interval or significance test.

In [ ]:
outer_results = phase_2c_results["outer_fold_results"]
inner_results = phase_2c_results["inner_candidate_results"]
outer_summary = phase_2c_results["outer_summary"]
selection_frequency = phase_2c_results["selection_frequency"]

display(outer_results)
display(inner_results)
display(outer_summary)
display(selection_frequency)


## 6. Export experiment evidence

Upload these four CSV files for review. The notebook does not evaluate the historical holdout and does not declare a final production model.

In [ ]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
outputs = {
    "phase_2c_outer_fold_results.csv": outer_results,
    "phase_2c_inner_candidate_results.csv": inner_results,
    "phase_2c_outer_summary.csv": outer_summary,
    "phase_2c_selection_frequency.csv": selection_frequency,
}
for filename, frame in outputs.items():
    frame.to_csv(REPORTS_DIR / filename, index=False)
    print(f"Saved: {REPORTS_DIR / filename}")
print("Training-only experiment complete. Historical holdout remains unused.")


## Phase 2C completion checklist

- [ ] Run all cells successfully in Colab.
- [ ] Confirm five outer folds and fifteen inner candidate records.
- [ ] Confirm no saved error outputs.
- [ ] Upload all four generated CSV files for review.
- [ ] Review outer-fold performance, parameter stability, and candidate-selection frequency.
- [ ] Record the conclusion without using the historical holdout.

**Interpretation rule:** if the tuned procedure does not show a stable validation improvement, retain the simpler existing configuration. Do not claim statistical significance from five outer folds.